# 4: Baseline Model: N-Gram Language Model
Apply a simple n-gram model to generate paper titles from abstracts and observe its output.

---

I chose a bigram language model as the baseline.

It estimates the probability of a word given the previous word:

$$P(w_n \mid w_{n-1}) = \frac{C(w_{n-1},\ w_n)}{C(w_{n-1})}$$

w/ Laplace (add-1) smoothing to handle unseen word pairs

## Imports

In [12]:
import re
import pandas as pd
from collections import Counter, defaultdict

import nltk
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab", quiet=True)

True

## Loading Data

In [19]:
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../ext/sample_50k_preprocessed.parquet")

train_df, temp_df = train_test_split(df, test_size=0.2,  random_state=42)
val_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,}")
print(f"Test: {len(test_df):,}")

Train: 40,000
Test: 5,000


## Cleaning Text

In [16]:
LATEX_PATTERNS = [
    (r"\$[^$]*\$", ""),
    (r"\\[a-zA-Z]+\{[^}]*\}", ""),
    (r"\\[a-zA-Z]+", ""),
    (r"\{[^}]*\}", ""),
    (r"\s+", " "),
]

In [17]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    for pattern, replacement in LATEX_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text.encode("ascii", errors="ignore").decode().strip().lower()

for split in [train_df, test_df]:
    split["abstract_clean"] = split["abstract"].apply(clean_text)
    split["title_clean"]    = split["title"].apply(clean_text)

## Build Bigram Model:

In [22]:
unigram_counts = Counter()
bigram_counts  = defaultdict(Counter)

for title in train_df["title_clean"]:
    tokens = ["<s>"] + word_tokenize(title) + ["</s>"]
    for token in tokens:
        unigram_counts[token] += 1
    for w1, w2 in zip(tokens[:-1], tokens[1:]):
        bigram_counts[w1][w2] += 1

VOCAB_SIZE = len(unigram_counts)
print(f"Vocabulary size: {VOCAB_SIZE:,}")
print(f"Unique bigrams: {sum(len(v) for v in bigram_counts.values()):,}")

#top 10 of the most common title-starting words
for word, count in bigram_counts["<s>"].most_common(10):
    print(f"{word:<20}: {count:,}")

Vocabulary size: 49,920
Unique bigrams: 247,360
A                   : 2,304
The                 : 2,010
On                  : 1,410
An                  : 454
Quantum             : 337
Towards             : 174
Learning            : 173
Efficient           : 147
Optimal             : 133
New                 : 128


## Generating Titles

In [23]:
def generate_title_ngram(max_len=12):
    current = "<s>"
    title   = []
    for _ in range(max_len):
        if current not in bigram_counts or len(bigram_counts[current]) == 0:
            break
        next_word = bigram_counts[current].most_common(1)[0][0]
        if next_word == "</s>":
            break
        title.append(next_word)
        current = next_word
    return " ".join(title)

print("Sample outputs:\n")
for i in range(10):
    row = test_df.iloc[i]
    print(f"[{i+1}]")
    print(f"Abstract  : {row['abstract_clean'][:120]}...")
    print(f"Reference : {row['title']}")
    print(f"Generated : {generate_title_ngram()}")
    print()

Sample outputs:

[1]
Abstract  : We present a superconvergent hybridizable discontinuous Galerkin (HDG) method for the steady-state incompressible Navier...
Reference : A superconvergent HDG method for the Incompressible Navier-Stokes
  Equations on general polyhedral meshes
Generated : A new physics

[2]
Abstract  : We study the asymptotic dynamics of stochastic Young differential delay equations under the regular assumptions on Lipsc...
Reference : Pullback attractors for stochastic Young differential delay equations
Generated : A new physics

[3]
Abstract  : The process of hand washing, according to the WHO, is divided into stages with clearly defined two handed dynamic gestur...
Reference : Tracking Hand Hygiene Gestures with Leap Motion Controller
Generated : A new physics

[4]
Abstract  : New corrections to the energy of -levels of positronium of order which are as large as several hundred kilohertz are obt...
Reference : New Corrections of Order $\alpha^6$ to $s$-Levels of Two-B

## Observations

The bigram model generates the **same title every time** regardless of what the abstract says. This is its core limitation — greedy decoding always follows the most probable path, which converges to a single repeated sequence of common academic words.

This was expected and tells us the following:

- The model has **no understanding of the input**, it cannot connect abstract content to title content
- It captures **surface-level statistics** only, what words commonly appear in titles
- It demonstrates why we need models that can actually **read and summarize** the abstract

This just serves as a baseline for more complex algorithms